# 02 - 对角矩阵优化与块重构

本 Notebook 演示：
1. Gamma 矫正（块级对角矩阵优化）的数学推导与验证
2. 单块重构效果
3. 全局训练集重构
4. 重构前后图像对比
5. 重构质量统计

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

from data.load_dataset import load_orl, train_test_split_orl
from src.biclustering import BidirectionalClustering
from src.reconstruction import (
    optimize_diagonal, reconstruct_block,
    reconstruct_all_blocks, reconstruction_error
)
from src.utils import show_reconstruction_comparison

X, y = load_orl(data_dir='../data/ORL')
X_train, X_test, y_train, y_test = train_test_split_orl(X, y, n_train=5)
IMG_SHAPE = (64, 64)
print(f'训练集: {X_train.shape}')

训练集: (200, 4096)


## 1. 单块的 Gamma 矫正（对角矩阵优化）

In [2]:
# 双向聚类
bicluster = BidirectionalClustering(n_row_clusters=4, n_col_clusters=4, random_state=42)
bicluster.fit(X_train)
blocks, row_idx_map, col_idx_map = bicluster.get_blocks(X_train)

# 选取最大块做演示
block_sizes = {k: v.shape[0] * v.shape[1] for k, v in blocks.items()}
a, b = max(block_sizes, key=block_sizes.get)
X_block = blocks[(a, b)]
print(f'最大块 ({a},{b}): shape={X_block.shape}')

最大块 (3,3): shape=(69, 1621)


In [3]:
# optimize_diagonal 返回 (l_a, r_b, mu, gamma) - 4个值
l_a, r_b, mu, gamma = optimize_diagonal(X_block)

print(f'最优 Gamma: {gamma:.4f}')
print(f'块均值: {X_block.mean():.4f} -> Gamma 矫正后均值: {np.power(X_block + 1e-8, gamma).mean():.4f}')
print(f'左权重 l_a: shape={l_a.shape}, min={l_a.min():.4f}, max={l_a.max():.4f}')
print(f'右权重 r_b: shape={r_b.shape}, min={r_b.min():.4f}, max={r_b.max():.4f}')

# 重构: X_hat = X^gamma (element-wise)
X_block_hat = reconstruct_block(X_block, gamma)
print(f' 重构块: shape={X_block_hat.shape}, range=[{X_block_hat.min():.3f}, {X_block_hat.max():.3f}]')

err = reconstruction_error(X_block, X_block_hat)
print(f'重构误差: 相对={err["relative"]:.4f}, PSNR={err["psnr"]:.2f} dB')

最优 Gamma: 1.2518
块均值: 0.5748 -> Gamma 矫正后均值: 0.5051
左权重 l_a: shape=(69,), min=0.8317, max=0.9229
右权重 r_b: shape=(1621,), min=0.7335, max=0.9281
 重构块: shape=(69, 1621), range=[0.022, 0.951]
重构误差: 相对=0.1193, PSNR=23.01 dB


In [4]:
# 可视化 Gamma 矫正效果
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].bar(range(len(l_a)), l_a, color='#4c72b0')
axes[0].set_xlabel('Sample index (within cluster)')
axes[0].set_ylabel('l_a')
axes[0].set_title(f'Sample Gamma Effect (Cluster {a})')
axes[0].axhline(y=1.0, color='red', linestyle='--', alpha=0.5)

axes[1].plot(r_b[:200], color='#dd8452', linewidth=0.8)
axes[1].set_xlabel('Feature index')
axes[1].set_ylabel('r_b')
axes[1].set_title(f'Feature Gamma Effect (first 200 dims)')
axes[1].axhline(y=1.0, color='red', linestyle='--', alpha=0.5)

z_flat = X_block.flatten()
z_gamma = X_block_hat.flatten()
axes[2].scatter(z_flat[::5], z_gamma[::5], alpha=0.3, s=3, color='#55a868')
axes[2].plot([0, 1], [0, 1], 'r--', alpha=0.5, label='y=x')
axes[2].set_xlabel('Original pixel value')
axes[2].set_ylabel(f'Gamma-corrected (gamma={gamma:.2f})')
axes[2].set_title('Pixel Transformation')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../results/figures/02_diagonal_weights.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\86175\AppData\Local\Temp\ipykernel_37488\989044303.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. 全局训练集重构

In [5]:
X_hat_train, block_params = reconstruct_all_blocks(
    X_train, blocks, row_idx_map, col_idx_map
)

err_global = reconstruction_error(X_train, X_hat_train)
print('全局重构误差:')
for k, v in err_global.items():
    print(f'  {k}: {v:.4f}')

print(' 各块 Gamma 值:')
for (i, j), params in sorted(block_params.items()):
    g = params['gamma']
    n_rows = len(params['row_idx'])
    n_cols = len(params['col_idx'])
    print(f'  Block ({i},{j}): gamma={g:.3f}, size={n_rows}x{n_cols}')

全局重构误差:
  frobenius: 80.7490
  relative: 0.1558
  mse: 0.0080
  psnr: 20.9912
 各块 Gamma 值:
  Block (0,0): gamma=0.983, size=19x1069
  Block (0,1): gamma=1.652, size=19x1160
  Block (0,2): gamma=0.740, size=19x246
  Block (0,3): gamma=1.199, size=19x1621
  Block (1,0): gamma=0.903, size=51x1069
  Block (1,1): gamma=1.118, size=51x1160
  Block (1,2): gamma=0.803, size=51x246
  Block (1,3): gamma=0.900, size=51x1621
  Block (2,0): gamma=1.123, size=61x1069
  Block (2,1): gamma=1.587, size=61x1160
  Block (2,2): gamma=0.598, size=61x246
  Block (2,3): gamma=1.153, size=61x1621
  Block (3,0): gamma=1.040, size=69x1069
  Block (3,1): gamma=1.873, size=69x1160
  Block (3,2): gamma=0.624, size=69x246
  Block (3,3): gamma=1.252, size=69x1621


In [6]:
fig = show_reconstruction_comparison(
    X_train, X_hat_train, y_train,
    img_shape=IMG_SHAPE,
    n_samples=8,
    title='Training Set: Original vs Gamma-Reconstructed Faces',
    save_path='../results/figures/02_reconstruction_comparison.png'
)
plt.show()

C:\Users\86175\AppData\Local\Temp\ipykernel_37488\4185965932.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
diff = np.abs(X_train - X_hat_train)
fig, axes = plt.subplots(3, 5, figsize=(12, 7))
for col in range(5):
    axes[0, col].imshow(X_train[col].reshape(IMG_SHAPE), cmap='gray', vmin=0, vmax=1)
    axes[0, col].axis('off')
    axes[0, col].set_title(f'S{y_train[col]+1}', fontsize=8)
    axes[1, col].imshow(X_hat_train[col].reshape(IMG_SHAPE), cmap='gray', vmin=0, vmax=1)
    axes[1, col].axis('off')
    axes[2, col].imshow(diff[col].reshape(IMG_SHAPE), cmap='hot', vmin=0)
    axes[2, col].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=9)
axes[1, 0].set_ylabel('Reconstructed', fontsize=9)
axes[2, 0].set_ylabel('|Difference|', fontsize=9)
fig.suptitle('Reconstruction Quality (Gamma Correction)', fontsize=12)
plt.tight_layout()
plt.savefig('../results/figures/02_reconstruction_diff.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\86175\AppData\Local\Temp\ipykernel_37488\4243264734.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
